In [1]:
import os
from pathlib import Path
import pandas as pd
from tqdm import tqdm

# 您的 SWC 文件路径
SWC_DIR = r"J:\BLA_four_types\csv_PFC_ways\swc"

def scan_swc_issues(folder_path):
    folder = Path(folder_path)
    swc_files = list(folder.glob("**/*.swc"))
    
    if not swc_files:
        print(f"❌ 未在路径找到 SWC 文件: {folder_path}")
        return

    print(f"🔍 找到 {len(swc_files)} 个 SWC 文件，开始检查拓扑结构...\n")
    
    problematic_files = []
    
    for file_path in tqdm(swc_files, desc="扫描中"):
        try:
            data_rows = []
            with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
                for line in f:
                    line = line.strip()
                    if line and not line.startswith("#"):
                        data_rows.append(line.split()[:7]) # 取前7列
            
            if not data_rows:
                continue

            df = pd.DataFrame(data_rows, columns=["id", "type", "x", "y", "z", "radius", "parent"])
            df["id"] = df["id"].astype(int)
            df["parent"] = df["parent"].astype(int)
            
            valid_ids = set(df["id"].values)
            
            # 查找悬空节点：parent 不存在且不是根节点(-1)
            bad_nodes = df[~df["parent"].isin(valid_ids) & (df["parent"] != -1)]
            
            if not bad_nodes.empty:
                issue_info = {
                    "file_name": file_path.name,
                    "file_path": str(file_path),
                    "bad_nodes": bad_nodes[["id", "parent"]].to_dict(orient="records")
                }
                problematic_files.append(issue_info)
                
        except Exception as e:
            print(f"\n⚠️ 读取文件失败: {file_path.name} | 错误: {e}")

    # 输出排查结果
    print("\n" + "=" * 60)
    print(f"📊 扫描完成！共发现 {len(problematic_files)} 个存在异常节点的文件：")
    print("=" * 60)
    
    if not problematic_files:
        print("✅ 所有 SWC 文件的父子节点拓扑关系均正常，未发现悬空节点！")
        return

    for idx, item in enumerate(problematic_files, 1):
        print(f"\n[{idx}] 问题文件: {item['file_name']}")
        print(f"    完整路径: {item['file_path']}")
        print(f"    异常节点数: {len(item['bad_nodes'])}")
        print("    具体悬空节点详情 (前 5 个):")
        for node in item["bad_nodes"][:5]:
            print(f"      - 节点 ID: {node['id']} ➔ 指向了不存在的 Parent ID: {node['parent']}")
        if len(item["bad_nodes"]) > 5:
            print(f"      ... 还有 {len(item['bad_nodes']) - 5} 个异常节点")

if __name__ == "__main__":
    scan_swc_issues(SWC_DIR)

🔍 找到 17 个 SWC 文件，开始检查拓扑结构...



扫描中: 100%|██████████| 17/17 [00:00<00:00, 126.88it/s]


📊 扫描完成！共发现 0 个存在异常节点的文件：
✅ 所有 SWC 文件的父子节点拓扑关系均正常，未发现悬空节点！


In [3]:
from pathlib import Path
import pandas as pd
from tqdm import tqdm
import os

# 输入与输出路径
INPUT_DIR = r"J:\BLA_four_types\csv_PFC_ways\swc"
OUTPUT_DIR = r"J:\BLA_four_types\csv_PFC_ways\swc_reindexed"
os.mkdir(OUTPUT_DIR)

def reindex_swc_files(input_dir, output_dir):
    in_path = Path(input_dir)
    out_path = Path(output_dir)
    out_path.mkdir(parents=True, exist_ok=True)
    
    swc_files = list(in_path.glob("*.swc"))
    print(f"找到 {len(swc_files)} 个 SWC 文件，开始重编号...")

    for file_path in tqdm(swc_files, desc="重编号进度"):
        comments = []
        data_rows = []
        
        # 1. 分离注释与数据
        with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
            for line in f:
                line_str = line.strip()
                if not line_str:
                    continue
                if line_str.startswith("#"):
                    comments.append(line)
                else:
                    data_rows.append(line_str.split()[:7])

        if not data_rows:
            continue

        # 2. 构建 DataFrame
        df = pd.DataFrame(
            data_rows, 
            columns=["id", "type", "x", "y", "z", "radius", "parent"]
        )
        
        old_ids = df["id"].astype(int).tolist()
        parents = df["parent"].astype(int).tolist()

        # 3. 建立旧 ID -> 连续新 ID (1, 2, 3...) 的映射字典
        id_map = {old_id: new_id for new_id, old_id in enumerate(old_ids, start=1)}
        id_map[-1] = -1  # 保持根节点为 -1

        # 4. 更新 ID 和 Parent
        df["id"] = [id_map[i] for i in old_ids]
        df["parent"] = [id_map.get(p, -1) for p in parents]

        # 5. 保存到新目录
        target_file = out_path / file_path.name
        with open(target_file, "w", encoding="utf-8") as f:
            f.writelines(comments)
            df.to_csv(f, sep=" ", header=False, index=False)

    print(f"\n✅ 处理完成！连续编号的 SWC 文件已保存至：\n{out_path.resolve()}")

if __name__ == "__main__":
    reindex_swc_files(INPUT_DIR, OUTPUT_DIR)

找到 17 个 SWC 文件，开始重编号...


重编号进度: 100%|██████████| 17/17 [00:00<00:00, 48.86it/s]


✅ 处理完成！连续编号的 SWC 文件已保存至：
J:\BLA_four_types\csv_PFC_ways\swc_reindexed
